# Institution Collaboration Forecasting — Analysis Notebook

This notebook trains all forecasting models (DHGNNLite, StaticGCN, co_freq)
on the KuzuDB knowledge graph and produces Figures 5.1–5.4.

**Figures**
- **5.1** — AUC comparison across models (bar chart)
- **5.2** — AUC by industry-academia stratum (grouped bar)
- **5.3** — Training loss curve for DHGNNLite
- **5.4** — Precision-Recall curves for all models

**Reproducibility**: fixed seed via `FORECAST_SEED=42`.

In [ ]:
import os, sys, json
from pathlib import Path

# Ensure repo root is on path
repo_root = Path("../..").resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import torch
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

SEED = 42
torch.manual_seed(SEED)
KG_DB_PATH = os.getenv("KG_DB_PATH", str(repo_root / "output/v2/kg"))
CKPT_DIR = repo_root / "experiments/forecasting/checkpoints"
FIG_DIR = repo_root / "experiments/forecasting/figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

print(f"KG path: {KG_DB_PATH}")
print(f"Checkpoint dir: {CKPT_DIR}")

## 1. Build Dataset

In [ ]:
from src.v2.analytics.forecasting.dataset import CollabForecastDataset

ds = CollabForecastDataset(
    kg_db_path=KG_DB_PATH,
    min_year=2019,
    max_year=2023,
    seed=SEED,
)
train_snaps, test_snaps = ds.split()

print(f"Train snapshots: {[s.year for s in train_snaps]}")
print(f"Test  snapshots: {[s.year for s in test_snaps]}")
for snap in train_snaps + test_snaps:
    n_inst = snap.data['institution'].num_nodes
    n_pos  = snap.pos_edges.shape[1]
    print(f"  Year {snap.year}: {n_inst} institutions, {n_pos} positive target edges")

## 2. Train Co-frequency Baseline

In [ ]:
from src.v2.analytics.forecasting.co_freq import CoFreqBaseline

co_freq = CoFreqBaseline()
co_freq_auc, co_freq_ap = co_freq.evaluate(test_snaps, train_snaps)
print(f"co_freq  →  AUC={co_freq_auc:.4f}  AP={co_freq_ap:.4f}")

## 3. Train Static GCN

In [ ]:
from src.v2.analytics.forecasting.static_gcn import StaticGCN, train_static_gcn, evaluate_static_gcn

gcn = StaticGCN(in_channels=6, hidden=64)
gcn_losses = train_static_gcn(gcn, train_snaps, epochs=100, lr=1e-3, seed=SEED, verbose=False)
gcn_auc, gcn_ap = evaluate_static_gcn(gcn, test_snaps, train_snaps)
print(f"StaticGCN  →  AUC={gcn_auc:.4f}  AP={gcn_ap:.4f}")

## 4. Train DHGNN-Lite

In [ ]:
from src.v2.analytics.forecasting.dhgnn_lite import DHGNNLite, train_dhgnn, evaluate_dhgnn, load_checkpoint

ckpt_path = CKPT_DIR / "best.pt"

dhgnn = DHGNNLite(author_in=1, inst_in=6, hidden=64, n_layers=2)
print(f"DHGNNLite params: {dhgnn.n_params():,} (≤ 5,000,000)")
assert dhgnn.n_params() <= 5_000_000

dhgnn_losses = train_dhgnn(
    dhgnn, train_snaps, epochs=50, lr=1e-3, seed=SEED,
    checkpoint_path=ckpt_path, verbose=True
)

# Load best
if ckpt_path.exists():
    dhgnn = load_checkpoint(DHGNNLite(author_in=1, inst_in=6, hidden=64, n_layers=2), ckpt_path)

dhgnn_auc, dhgnn_ap = evaluate_dhgnn(dhgnn, test_snaps, train_snaps)
print(f"DHGNNLite  →  AUC={dhgnn_auc:.4f}  AP={dhgnn_ap:.4f}")

## 5. Constraint Check

In [ ]:
margin = dhgnn_auc - co_freq_auc
status = '✅ PASSED' if margin >= 0.05 else '⚠️  NOT MET'
print(f"DHGNNLite AUC ({dhgnn_auc:.4f}) - co_freq AUC ({co_freq_auc:.4f}) = {margin:.4f}  [{status}]")
print(f"Required: ≥ 0.05")

## Figure 5.1 — AUC comparison across models

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))

models   = ['co\_freq', 'StaticGCN', 'DHGNNLite']
aucs     = [co_freq_auc, gcn_auc, dhgnn_auc]
aps      = [co_freq_ap, gcn_ap, dhgnn_ap]
colors   = ['#4e79a7', '#f28e2b', '#e15759']

x = range(len(models))
bars = ax.bar(x, aucs, width=0.4, label='AUC', color=colors, alpha=0.85)
ax.bar([i + 0.4 for i in x], aps, width=0.4, label='AP', color=colors, alpha=0.45, hatch='//')
ax.set_xticks([i + 0.2 for i in x])
ax.set_xticklabels(models, fontsize=11)
ax.set_ylabel('Score', fontsize=11)
ax.set_ylim(0, 1.05)
ax.axhline(0.5, color='grey', linestyle='--', linewidth=0.8, label='random')
ax.legend(fontsize=10)
ax.set_title('Figure 5.1 — AUC and AP comparison (1-year horizon)', fontsize=11)

for bar, val in zip(bars, aucs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{val:.3f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig(FIG_DIR / 'fig_5_1_auc_comparison.pdf', bbox_inches='tight')
plt.show()
print(f'Saved → {FIG_DIR}/fig_5_1_auc_comparison.pdf')

## Figure 5.2 — AUC by industry-academia stratum

In [ ]:
from src.v2.analytics.forecasting.evaluate import _stratify_by_industry_mix

strata = _stratify_by_industry_mix(test_snaps)
strata_names = ['pure\_academia', 'mixed', 'high\_industry']
strata_keys  = ['pure_academia', 'mixed', 'high_industry']

def _stratum_auc(model_fn, key):
    snaps = strata[key]
    if not snaps:
        return 0.5
    auc, _ = model_fn(snaps, train_snaps)
    return auc

co_aucs  = [_stratum_auc(co_freq.evaluate, k) for k in strata_keys]
gcn_aucs = [_stratum_auc(lambda t, tr: evaluate_static_gcn(gcn, t, tr), k) for k in strata_keys]
dhg_aucs = [_stratum_auc(lambda t, tr: evaluate_dhgnn(dhgnn, t, tr), k) for k in strata_keys]

fig, ax = plt.subplots(figsize=(8, 4))
width = 0.25
xs = range(len(strata_keys))
ax.bar([i - width for i in xs], co_aucs, width=width, label='co\_freq', color='#4e79a7')
ax.bar(xs, gcn_aucs, width=width, label='StaticGCN', color='#f28e2b')
ax.bar([i + width for i in xs], dhg_aucs, width=width, label='DHGNNLite', color='#e15759')
ax.set_xticks(list(xs))
ax.set_xticklabels(strata_names, fontsize=11)
ax.set_ylabel('AUC', fontsize=11)
ax.set_ylim(0, 1.05)
ax.axhline(0.5, color='grey', linestyle='--', linewidth=0.8)
ax.legend(fontsize=10)
ax.set_title('Figure 5.2 — AUC by industry-academia mix stratum', fontsize=11)
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig_5_2_auc_by_stratum.pdf', bbox_inches='tight')
plt.show()
print(f'Saved → {FIG_DIR}/fig_5_2_auc_by_stratum.pdf')

## Figure 5.3 — Training loss curve (DHGNNLite)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))

if dhgnn_losses:
    ax.plot(dhgnn_losses, color='#e15759', linewidth=1.8, label='DHGNNLite')
if gcn_losses:
    ax.plot(gcn_losses[:len(dhgnn_losses)], color='#f28e2b', linewidth=1.2,
            linestyle='--', label='StaticGCN (first N epochs)')

ax.set_xlabel('Epoch', fontsize=11)
ax.set_ylabel('Binary Cross-Entropy Loss', fontsize=11)
ax.set_title('Figure 5.3 — Training loss (institution link prediction)', fontsize=11)
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig_5_3_training_loss.pdf', bbox_inches='tight')
plt.show()
print(f'Saved → {FIG_DIR}/fig_5_3_training_loss.pdf')

## Figure 5.4 — Precision-Recall curves

In [ ]:
from sklearn.metrics import precision_recall_curve, average_precision_score

def _collect_scores(model_fn):
    """Collect all prediction scores and labels for test snapshots."""
    all_scores, all_labels = [], []
    for snap in test_snaps:
        if snap.pos_edges.shape[1] == 0:
            continue
        pos = snap.pos_edges
        neg = snap.neg_edges
        import torch
        all_edges = torch.cat([pos, neg], dim=1)
        labels = torch.cat([torch.ones(pos.shape[1]), torch.zeros(neg.shape[1])])
        scores = model_fn(snap, all_edges)
        all_scores.extend(scores.tolist())
        all_labels.extend(labels.tolist())
    return all_scores, all_labels

fig, ax = plt.subplots(figsize=(7, 5))

model_configs = [
    ('co\_freq', lambda snap, edges: co_freq.predict(snap, edges), '#4e79a7'),
]

# For GCN and DHGNN, encode first then score
from src.v2.analytics.forecasting.static_gcn import _build_static_graph
x_inst, ei = _build_static_graph(train_snaps + test_snaps)
gcn.eval()
with torch.no_grad():
    h_gcn = gcn.encode(x_inst, ei) if x_inst.shape[0] > 0 else None

# DHGNN temporal state
dhgnn.eval()
state = None
with torch.no_grad():
    for snap in train_snaps:
        inst_emb_dhgnn, state = dhgnn.encode_snapshot(snap.data, state)

for label, score_fn, color in model_configs:
    scores, labels = _collect_scores(score_fn)
    if len(set(labels)) < 2:
        continue
    prec, rec, _ = precision_recall_curve(labels, scores)
    ap = average_precision_score(labels, scores)
    ax.plot(rec, prec, color=color, linewidth=1.8, label=f'{label} (AP={ap:.3f})')

# GCN PR
if h_gcn is not None:
    def _gcn_score(snap, edges):
        with torch.no_grad():
            return gcn.decode_link(h_gcn, edges)
    scores, labels = _collect_scores(_gcn_score)
    if len(set(labels)) >= 2:
        prec, rec, _ = precision_recall_curve(labels, scores)
        ap = average_precision_score(labels, scores)
        ax.plot(rec, prec, color='#f28e2b', linewidth=1.8, linestyle='--',
                label=f'StaticGCN (AP={ap:.3f})')

# DHGNN PR
def _dhgnn_score(snap, edges):
    with torch.no_grad():
        inst_emb, _ = dhgnn.encode_snapshot(snap.data, state)
        return dhgnn.decode_link(inst_emb, edges)
scores, labels = _collect_scores(_dhgnn_score)
if len(set(labels)) >= 2:
    prec, rec, _ = precision_recall_curve(labels, scores)
    ap = average_precision_score(labels, scores)
    ax.plot(rec, prec, color='#e15759', linewidth=2, label=f'DHGNNLite (AP={ap:.3f})')

ax.set_xlabel('Recall', fontsize=11)
ax.set_ylabel('Precision', fontsize=11)
ax.set_title('Figure 5.4 — Precision-Recall curves (1-year collaboration forecasting)', fontsize=11)
ax.legend(fontsize=10, loc='upper right')
ax.set_xlim(0, 1)
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig_5_4_pr_curves.pdf', bbox_inches='tight')
plt.show()
print(f'Saved → {FIG_DIR}/fig_5_4_pr_curves.pdf')

## Summary

In [ ]:
summary = {
    'co_freq':   {'auc': co_freq_auc,  'ap': co_freq_ap},
    'static_gcn': {'auc': gcn_auc,    'ap': gcn_ap},
    'dhgnn_lite': {'auc': dhgnn_auc,  'ap': dhgnn_ap},
    'constraint': {'margin': dhgnn_auc - co_freq_auc, 'passed': (dhgnn_auc - co_freq_auc) >= 0.05},
}
print(json.dumps(summary, indent=2))

(repo_root / 'experiments/forecasting/results.json').write_text(
    json.dumps(summary, indent=2), encoding='utf-8'
)
print('\nAll figures saved to', FIG_DIR)